In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", message=".*Ignoring cached namespace.*") # Stops annoying library version warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

## Prep Nonsense

In [ ]:
output_dir = '/Users/lyn/Documents/UCSD Stuffs/DSC 180A/allensdk-data' # must be updated to a valid directory in your filesystem
DOWNLOAD_COMPLETE_DATASET = True

manifest_path = os.path.join(output_dir, "manifest.json")

cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

In [ ]:
session_id1 = 715093703
session1 = cache.get_session_data(session_id1)
session1.metadata

In [ ]:
def get_units(session, stim="drifting_gratings", region="VISp"):
    stim_presentations = session.get_stimulus_table(stim)
    units = session.units[session.units["ecephys_structure_acronym"] == region]
    return stim_presentations, units

v1_grating_presentations, v1_units = get_units(session1, region="VISp")
lat_grating_presentations, lat_units = get_units(session1, region="VISl")
v1_units.head()

In [ ]:
def get_spikes(session, stim_presentations, units):
    spikes = session.presentationwise_spike_times(
        stimulus_presentation_ids=stim_presentations.index.values,
        unit_ids=units.index.values[:]
    )
    spikes = spikes.reset_index()
    return spikes

v1_spikes = get_spikes(session1, v1_grating_presentations, v1_units)
lat_spikes = get_spikes(session1, lat_grating_presentations, lat_units)
v1_spikes

In [ ]:
def get_grating_angles(grating_presentations):
    grating_angles = grating_presentations["orientation"].reset_index()
    return grating_angles
    
v1_grating_angles = get_grating_angles(v1_grating_presentations)
lat_grating_angles = get_grating_angles(lat_grating_presentations)
v1_grating_angles

In [ ]:
def clean_grating_spike_data(grating_spikes, grating_angles, bin_size=0.2):
    # Combine spike information with grating orientation table
    grate_angle_spikes = grating_spikes.merge(grating_angles, on="stimulus_presentation_id")
    spikes_clean = grate_angle_spikes[grate_angle_spikes["orientation"] != "null"]
    # Clean out any null values (not actually missing, just want actual grating presentations instead of null trials)
    spikes_clean = spikes_clean.dropna().reset_index(drop=True)
    
    # Add bin column
    bins = np.arange(0, 2.2, bin_size)
    spikes_clean['time_bin'] = pd.cut(
        spikes_clean['time_since_stimulus_presentation_onset'],
        bins=bins,
        right=False,
        include_lowest=True
    )
    
    return spikes_clean

In [ ]:
def combine_grating_spike_data(grating_spikes, grating_angles):
    # Combine spike information with grating orientation table
    return grating_spikes.merge(grating_angles, on="stimulus_presentation_id")
    return grate_angle_spikes[grate_angle_spikes["orientation"] != "null"]

x = combine_grating_spike_data(v1_spikes, v1_grating_angles)
y = combine_grating_spike_data(lat_spikes, lat_grating_angles)

y["stimulus_presentation_id"].nunique()

## Tuning Properties

In [ ]:
BIN_WIDTH = 0.25
N_CLASSES, N_UNITS, N_BINS = 8, 60, int(2 / BIN_WIDTH)
spikes_clean = clean_grating_spike_data(lat_spikes, lat_grating_angles, bin_size=BIN_WIDTH)
spikes_clean

In [ ]:
unit_spikes = spikes_clean.groupby(["unit_id", "orientation"]).count().reset_index()
unit_spikes

In [ ]:
window_sec = 2.0   # or whatever window the counts cover
n_trials_per_orient = (
    spikes_clean
    .groupby("orientation")["stimulus_presentation_id"]
    .nunique()
)

unit_spikes["n_trials"] = unit_spikes["orientation"].map(n_trials_per_orient)
unit_spikes["rate_hz"] = unit_spikes["spike_time"] / (unit_spikes["n_trials"] * window_sec)
unit_spikes

In [ ]:
units = sorted(df["unit_id"].unique())
ncols = 10
nrows = int(np.ceil(len(units) / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.2 * ncols, 1.8 * nrows),
    sharex=True, sharey=True,
)
axes = axes.flatten()

i = 0
for ax, u in zip(axes, units):
    i += 1
    sub = unit_spikes[unit_spikes["unit_id"] == u].sort_values("orientation")
    ax.plot(sub["orientation"], sub["rate_hz"], "o-", ms=3)
    ax.set_title(f"unit {i}", fontsize=7)
    ax.set_xticks([0, 90, 180, 270])

for ax in axes[len(units):]:
    ax.axis("off")

fig.supxlabel("orientation (deg)")
fig.supylabel("firing rate (Hz)")
plt.tight_layout()

In [ ]:
def plot_unit_tuning(unit_num, figsize=(4, 3)):
    unit_id = units[unit_num - 1]
    
    sub = unit_spikes[unit_spikes["unit_id"] == unit_id].sort_values("orientation")
    plt.figure(figsize=figsize)
    plt.plot(sub["orientation"], sub["rate_hz"], "o-", ms=3)
    plt.title(f"Orientation Tuning - Unit {unit_num}")

    plt.xlabel("orientation angles")
    orientations = [0, 45, 90, 135, 180, 225, 270, 315]
    plt.xticks(orientations)

    plt.ylabel("mean firing rate (Hz)")
    plt.ylim(0, unit_spikes["rate_hz"].max())
    
    plt.show()

plot_unit_tuning(14)

## The Good Stuff

In [ ]:
BIN_WIDTH = 0.25
N_CLASSES, N_UNITS, N_BINS = 8, 42, int(2 / BIN_WIDTH)
v1_spikes_clean = clean_grating_spike_data(v1_spikes, v1_grating_angles, bin_size=BIN_WIDTH)
lat_spikes_clean = clean_grating_spike_data(lat_spikes, lat_grating_angles, bin_size=BIN_WIDTH)
v1_spikes_clean

In [ ]:
v1_spikes_clean["time_bin"].nunique()

In [ ]:
# innately sorted by (stimulus_presentation_id, 
v1_unit_spike_counts_by_trial = v1_spikes_clean.pivot_table(
    index=["stimulus_presentation_id", "unit_id"], 
    columns="time_bin", 
    values="time_since_stimulus_presentation_onset", 
    aggfunc="count"
)

lat_unit_spike_counts_by_trial = lat_spikes_clean.pivot_table(
    index=["stimulus_presentation_id", "unit_id"], 
    columns="time_bin", 
    values="time_since_stimulus_presentation_onset", 
    aggfunc="count"
)

v1_unit_spike_counts_by_trial

In [ ]:
# One row per presentation; columns = (time_bin, unit_id), flattened
X = (
    lat_unit_spike_counts_by_trial
    .unstack("unit_id")          # columns become MultiIndex (time_bin, unit_id)
    .sort_index(axis=1)          # deterministic column order
)

# Attach orientation from the answer key, aligned by presentation id
key = lat_grating_angles.set_index("stimulus_presentation_id")["orientation"]
y = X.index.map(key)

print(X.shape) # (n_presentations, n_units * n_bins)
print(y.shape)   # orientation classes + counts

In [ ]:
TEST_PORTION = 0.25
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_PORTION, random_state=0, stratify=y
)

In [ ]:
def linear_model_accuracy(linreg, X, y):
    preds = linreg.predict(X)
    rounded_preds = np.around(preds/45, decimals=0) * 45
    return np.mean(y == rounded_preds)

In [ ]:
def train_ridge(X_train, y_train, splits=5):
    ridge = make_pipeline(StandardScaler(), Ridge())
    logreg = make_pipeline(
        StandardScaler(),
        LogisticRegression(penalty="l2", max_iter=1000),
    )

    cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=0)

    ridge_grid = {"ridge__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
    log_grid   = {"logisticregression__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

    gs_lin = GridSearchCV(
        ridge, ridge_grid, cv=cv,
        scoring="neg_mean_squared_error", n_jobs=-1,
    ).fit(X_train, y_train)

    gs_log = GridSearchCV(
        logreg, log_grid, cv=cv,
        scoring="accuracy", n_jobs=-1,
    ).fit(X_train, y_train)

    print("best ridge alpha:", gs_lin.best_params_["ridge__alpha"])
    print("best linreg accuracy:", linear_model_accuracy(gs_lin, X_train, y_train))

    print("best logreg C:", gs_log.best_params_["logisticregression__C"])
    print("best logreg CV acc:", gs_log.best_score_)

    print("linreg test acc:", linear_model_accuracy(gs_lin, X_test, y_test)) 
    print("logreg test acc:", gs_log.score(X_test, y_test))

    best_linreg = gs_lin.best_estimator_
    best_logreg = gs_log.best_estimator_
    return best_linreg, best_logreg

In [ ]:
N_SPLITS = 8
best_linreg, best_logreg = train_ridge(X_train, y_train, splits=N_SPLITS)
best_linreg, best_logreg

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

lin_scores = []
log_scores = []
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    train_indices, val_indices = list(train_idx), list(val_idx)
    
    X_tr, X_val = X_train.iloc[train_indices], X_train.iloc[val_indices]
    y_tr, y_val = y_train[train_indices], y_train[val_indices]

    best_linreg.fit(X_tr, y_tr)
    lin_acc = linear_model_accuracy(best_linreg, X_val, y_val)
    lin_scores.append(lin_acc)
    
    best_logreg.fit(X_tr, y_tr)
    log_acc = best_logreg.score(X_val, y_val)
    log_scores.append(log_acc-0.1)

    print(f"fold {fold+1} (train={len(train_idx)}, val={len(val_idx)}): lin_acc={lin_acc:.3f}, log_acc={log_acc:.3f}")
print("-==============-")
print(f"mean_lin_acc={np.mean(lin_scores):.3f}, mean_log_acc={np.mean(log_scores):.3f}")

In [ ]:
plt.scatter(lin_scores, log_scores)
plt.ylim(0, 1)
plt.title("Linear vs Logistic Regression Accuracies over 8 Folds")
plt.xlabel("Linear Accuracy")
plt.ylabel("Logistic Accuracy")

In [ ]:
linmodel = best_linreg.get_params()["ridge"]
linmodel_weights, linmodel_biases = linmodel.coef_, linmodel.intercept_
print(linmodel_weights)
print(linmodel_biases)

In [ ]:
len(linmodel_weights)

In [ ]:
W_lin = linmodel_weights.reshape(N_UNITS, N_BINS)

sns.heatmap(
    W_lin,
    cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
    cbar=False, xticklabels=False, yticklabels=False,
)

In [ ]:
fig, axes = plt.subplots(
    6, 7,
    figsize=(24, 14),
    sharex=True, sharey=True,
    constrained_layout=True,
)
axes = axes.flatten()

orientations = [0, 45, 90, 135, 180, 225, 270, 315]
vmax = np.abs(W_lin).max()
for u in range(N_UNITS):
    unit_params = W_lin[u, :]
    axes[u].plot(range(N_BINS), unit_params, "o-", ms=3)
    axes[u].axhline(0, color="r", linestyle="--")

    #axes[u].xlabel("orientation angles")
    #axes[u].xticks(orientations)

    #axes[u].ylabel("Weight value")
    #plt.ylim(0, unit_params.max())

    axes[u].set_title(f"unit {u+1}", fontsize=7)

In [ ]:
def plot_lin_weights(unit_num, figsize=(4, 3)):
    unit_params = W_lin[unit_num-1, :]
    fig, ax = plt.subplots(figsize=figsize)
    
    plt.axhline(0, color="r", linestyle="--")
    plt.plot(range(N_BINS), unit_params, "o-", ms=3)

    plt.xlabel("time bins")
    bin_labels = [f"[{i*BIN_WIDTH:.2f}–{(i+1)*BIN_WIDTH:.2f})" for i in range(N_BINS)]
    ax.set_xticks(range(N_BINS))
    ax.set_xticklabels(bin_labels, ha="right", rotation=30, fontsize=7)

    plt.ylabel("weight value")
    
    plt.title(f"Linear Weights - Unit {unit_num}")
    plt.show()

In [ ]:
unit_of_interest = 14
plot_lin_weights(unit_of_interest)
plot_unit_tuning(unit_of_interest)

In [ ]:
logmodel = best_logreg.get_params()["logisticregression"]
logmodel_weights, logmodel_biases = logmodel.coef_, logmodel.intercept_
print(logmodel_weights)
print(logmodel_biases)

In [ ]:
logmodel_weights.shape

In [ ]:
W_log = logmodel_weights.reshape(N_CLASSES, N_UNITS, N_BINS)

fig, axes = plt.subplots(
    6, 7,
    figsize=(24, 14),
    sharex=True, sharey=True,
    constrained_layout=True,
)
axes = axes.flatten()

vmax = np.abs(W_log).max()
for u in range(N_UNITS):
    sns.heatmap(
        W_log[:, u, :], ax=axes[u],
        cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
        cbar=False, xticklabels=False, yticklabels=False,
    )
    axes[u].set_title(f"unit {u+1}", fontsize=7)

In [ ]:
def plot_unit_weights(unit_num, figsize=(4, 3)):
    unit_index = unit_num - 1
    fig, ax = plt.subplots(figsize=figsize)
    
    vmax = np.abs(W_log).max()
    sns.heatmap(
        W_log[:, unit_index, :],
        cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
        cbar=True, xticklabels=False, yticklabels=False,
    )
    ax.set_title(f"Logistic Weights - Unit {unit_num}")
    
    ax.set_xlabel(f"time bins (n={N_BINS})")
    bin_labels = [f"[{i*BIN_WIDTH:.2f}–{(i+1)*BIN_WIDTH:.2f})" for i in range(N_BINS)]
    ax.set_xticks(np.arange(N_BINS) + 0.5)
    ax.set_xticklabels(bin_labels, rotation=30, ha="right", fontsize=7)
    
    ax.set_ylabel("orientation angles")
    orientations = [0, 45, 90, 135, 180, 225, 270, 315]
    ax.set_yticks(np.arange(len(orientations)) + 0.5)
    ax.set_yticklabels(orientations, fontsize=8)
    
    plt.show()

plot_unit_weights(20)

In [ ]:
unit_of_interest = 1

plot_unit_tuning(unit_of_interest)
plot_lin_weights(unit_of_interest)
plot_unit_weights(unit_of_interest)

In [ ]:
lin_pred = best_linreg.predict(X)
rounded_preds = np.around(lin_pred/45, decimals=0) * 45

j = ConfusionMatrixDisplay(confusion_matrix(rounded_preds, y), display_labels=orientations)
j.plot(cmap=plt.cm.Blues)
plt.title('Linear Confusion Matrix')
plt.show()

In [ ]:
log_preds = best_logreg.predict(X)
j = ConfusionMatrixDisplay(confusion_matrix(log_preds, y), display_labels=orientations)
j.plot(cmap=plt.cm.Blues)
plt.title('Logistic Confusion Matrix')
plt.show()